# Install required packages
pip install spacy pandas

# Download German language model
python -m spacy download de_core_news_sm

python legal_metadata_extractor.py --test


In [ ]:
#!/usr/bin/env python3
"""
German Legal Document Metadata Extractor
Specialized for historical vocational education regulations

This prototype demonstrates automated metadata extraction from OCR'd German legal documents.
It uses multiple approaches: rule-based patterns, NER, and German language models.
"""

import re
import spacy
from datetime import datetime
from typing import Dict, List, Optional, Tuple
import pandas as pd
from pathlib import Path
import json

class GermanLegalMetadataExtractor:
    def __init__(self, model_name="de_core_news_sm"):
        """
        Initialize the metadata extractor for German legal documents

        Args:
            model_name (str): spaCy German model to use
        """
        self.model_name = model_name
        self.setup_models()
        self.setup_patterns()

    def setup_models(self):
        """Setup spaCy German NLP model"""
        try:
            self.nlp = spacy.load(self.model_name)
            print(f"✅ Loaded spaCy model: {self.model_name}")
        except OSError:
            print(f"❌ Model {self.model_name} not found. Install with:")
            print(f"python -m spacy download {self.model_name}")
            raise

    def setup_patterns(self):
        """Setup regex patterns for German legal documents"""

        # Title patterns - German legal documents often start with specific keywords
        self.title_patterns = [
            # Verordnung patterns
            r'(?i)^(.{0,50}(?:verordnung|ordnung|gesetz|bestimmung|vorschrift|regelung).{0,100})',
            # Bekanntmachung patterns
            r'(?i)^(.{0,50}(?:bekanntmachung|mitteilung|anordnung).{0,100})',
            # Decree patterns
            r'(?i)^(.{0,50}(?:erlass|verfügung|beschluss).{0,100})',
            # General title pattern (first significant line)
            r'^([A-ZÄÖÜ].{10,200}?)(?:\n|$)',
        ]

        # Date patterns - German date formats
        self.date_patterns = [
            # Day.Month.Year formats
            r'(?i)(?:vom\s+)?(\d{1,2}\.\s*\d{1,2}\.\s*\d{4})',
            r'(?i)(?:am\s+)?(\d{1,2}\.\s*\d{1,2}\.\s*\d{4})',
            # Month Year formats
            r'(?i)((?:januar|februar|märz|april|mai|juni|juli|august|september|oktober|november|dezember)\s+\d{4})',
            # Year only
            r'(?i)(?:jahr\s+)?(\d{4})',
            # Alternative formats
            r'(?i)(?:den\s+)?(\d{1,2}\.\s*\d{1,2}\.\s*\d{2,4})',
        ]

        # Publisher/Authority patterns
        self.publisher_patterns = [
            # Government authorities
            r'(?i)((?:bundes|landes|ministerium|behörde|amt|verwaltung).{0,100})',
            r'(?i)((?:regierung|senat|kammer).{0,50})',
            # Educational authorities
            r'(?i)((?:kultus|bildungs|schul).{0,50}(?:ministerium|behörde|amt))',
            # States
            r'(?i)((?:bayern|baden-württemberg|hessen|niedersachsen|nordrhein-westfalen|rheinland-pfalz|schleswig-holstein|hamburg|bremen|berlin|saarland|brandenburg|mecklenburg-vorpommern|sachsen|sachsen-anhalt|thüringen).{0,50})',
        ]

        # Author patterns (often officials or ministers)
        self.author_patterns = [
            # Official titles
            r'(?i)((?:minister|staatssekretär|direktor|präsident|senator).{0,100})',
            # Names with titles
            r'(?i)((?:dr\.?\s+|prof\.?\s+)?[A-ZÄÖÜ][a-zäöüß]+\s+[A-ZÄÖÜ][a-zäöüß]+)',
            # Signatures
            r'(?i)(?:gez\.?\s+|gezeichnet\s+|unterschrift\s+)([A-ZÄÖÜ].{0,50})',
        ]

        # Document type patterns
        self.document_type_patterns = [
            r'(?i)(berufsbildungsgesetz|ausbildungsordnung|prüfungsordnung|lehrplan)',
            r'(?i)(handwerksordnung|berufsschulordnung|weiterbildungsordnung)',
            r'(?i)(verordnung|gesetz|bestimmung|vorschrift|richtlinie)',
        ]

    def clean_text(self, text: str) -> str:
        """Clean OCR text for better processing"""
        # Remove excessive whitespace
        text = re.sub(r'\s+', ' ', text)
        # Remove page numbers and artifacts
        text = re.sub(r'(?i)seite\s+\d+', '', text)
        text = re.sub(r'\d+\s*$', '', text, flags=re.MULTILINE)
        return text.strip()

    def extract_title(self, text: str) -> Optional[str]:
        """Extract document title using multiple strategies"""
        candidates = []

        # Try regex patterns
        for pattern in self.title_patterns:
            matches = re.findall(pattern, text, re.MULTILINE | re.DOTALL)
            if matches:
                candidates.extend(matches)

        # Use spaCy for sentence extraction from beginning
        doc = self.nlp(text[:1000])  # First 1000 chars
        sentences = [sent.text.strip() for sent in doc.sents]

        # Look for title-like sentences (capitalized, substantial length)
        for sent in sentences[:5]:  # Check first 5 sentences
            if (len(sent) > 20 and
                sent[0].isupper() and
                not sent.lower().startswith(('der', 'die', 'das', 'ein', 'eine'))):
                candidates.append(sent)

        # Score candidates
        if candidates:
            # Prefer longer, more formal titles
            scored = [(len(c), c) for c in candidates if len(c) > 15]
            if scored:
                return max(scored)[1][:200]  # Limit length

        return None

    def extract_dates(self, text: str) -> List[str]:
        """Extract dates from text"""
        dates = []

        for pattern in self.date_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            dates.extend(matches)

        # Clean and validate dates
        valid_dates = []
        for date in dates:
            # Basic validation for years
            if re.search(r'19\d{2}|20\d{2}', date):
                valid_dates.append(date.strip())

        return list(set(valid_dates))  # Remove duplicates

    def extract_publishers(self, text: str) -> List[str]:
        """Extract publisher/authority information"""
        publishers = []

        for pattern in self.publisher_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            publishers.extend(matches)

        # Use NER for organization detection
        doc = self.nlp(text)
        for ent in doc.ents:
            if ent.label_ in ['ORG', 'MISC'] and len(ent.text) > 5:
                publishers.append(ent.text)

        # Clean and deduplicate
        cleaned = []
        for pub in publishers:
            pub = pub.strip()
            if len(pub) > 5 and pub not in cleaned:
                cleaned.append(pub)

        return cleaned[:5]  # Return top 5

    def extract_authors(self, text: str) -> List[str]:
        """Extract author information"""
        authors = []

        # Pattern-based extraction
        for pattern in self.author_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            authors.extend(matches)

        # NER-based person extraction
        doc = self.nlp(text)
        for ent in doc.ents:
            if ent.label_ == 'PER':
                authors.append(ent.text)

        # Clean and validate
        valid_authors = []
        for author in authors:
            author = author.strip()
            # Basic validation: should contain at least one uppercase letter
            if len(author) > 3 and re.search(r'[A-ZÄÖÜ]', author):
                valid_authors.append(author)

        return list(set(valid_authors))[:3]  # Return top 3 unique

    def extract_document_type(self, text: str) -> Optional[str]:
        """Extract document type"""
        for pattern in self.document_type_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            if matches:
                return matches[0]
        return None

    def extract_year(self, dates: List[str]) -> Optional[int]:
        """Extract most likely publication year"""
        years = []
        for date in dates:
            year_match = re.search(r'(19\d{2}|20\d{2})', date)
            if year_match:
                years.append(int(year_match.group(1)))

        if years:
            # Return most recent year (assuming it's publication date)
            return max(years)
        return None

    def extract_metadata(self, text: str, filename: str = None) -> Dict:
        """
        Extract all metadata from document text

        Args:
            text (str): OCR'd document text
            filename (str): Optional filename for context

        Returns:
            Dict: Extracted metadata
        """
        # Clean text first
        clean_text = self.clean_text(text)

        # Extract all metadata
        metadata = {
            'filename': filename,
            'title': self.extract_title(clean_text),
            'dates': self.extract_dates(clean_text),
            'publishers': self.extract_publishers(clean_text),
            'authors': self.extract_authors(clean_text),
            'document_type': self.extract_document_type(clean_text),
            'year': None,
            'confidence_score': 0.0
        }

        # Extract year from dates
        if metadata['dates']:
            metadata['year'] = self.extract_year(metadata['dates'])

        # Calculate confidence score
        metadata['confidence_score'] = self.calculate_confidence(metadata)

        return metadata

    def calculate_confidence(self, metadata: Dict) -> float:
        """Calculate confidence score for extracted metadata"""
        score = 0.0
        weights = {
            'title': 0.3,
            'year': 0.2,
            'publishers': 0.2,
            'authors': 0.15,
            'document_type': 0.15
        }

        for field, weight in weights.items():
            if field in metadata and metadata[field]:
                if isinstance(metadata[field], list):
                    score += weight if len(metadata[field]) > 0 else 0
                else:
                    score += weight

        return round(score, 2)

    def process_batch(self, texts: List[str], filenames: List[str] = None) -> List[Dict]:
        """Process multiple documents"""
        results = []

        if filenames is None:
            filenames = [f"document_{i+1}" for i in range(len(texts))]

        for i, text in enumerate(texts):
            try:
                metadata = self.extract_metadata(text, filenames[i])
                results.append(metadata)
                print(f"✅ Processed: {filenames[i]}")
            except Exception as e:
                print(f"❌ Error processing {filenames[i]}: {e}")
                results.append({
                    'filename': filenames[i],
                    'error': str(e),
                    'confidence_score': 0.0
                })

        return results

    def export_results(self, results: List[Dict], output_path: str, format: str = 'json'):
        """Export results to file"""
        if format == 'json':
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
        elif format == 'csv':
            # Flatten results for CSV
            flattened = []
            for result in results:
                flat = {
                    'filename': result.get('filename', ''),
                    'title': result.get('title', ''),
                    'year': result.get('year', ''),
                    'document_type': result.get('document_type', ''),
                    'confidence_score': result.get('confidence_score', 0.0),
                    'dates': '; '.join(result.get('dates', [])),
                    'publishers': '; '.join(result.get('publishers', [])),
                    'authors': '; '.join(result.get('authors', [])),
                    'error': result.get('error', '')
                }
                flattened.append(flat)

            df = pd.DataFrame(flattened)
            df.to_csv(output_path, index=False, encoding='utf-8')

        print(f"📄 Results exported to: {output_path}")


# Example usage and testing
def test_extractor():
    """Test the metadata extractor with sample German legal text"""

    # Sample German legal document text (simulated OCR output)
    sample_text = """
    Verordnung über die Berufsausbildung zum Fachinformatiker und zur Fachinformatikerin

    Vom 28. Februar 2020

    Auf Grund des § 4 Absatz 1 des Berufsbildungsgesetzes, der zuletzt durch Artikel 436
    der Verordnung vom 31. August 2015 (BGBl. I S. 1474) geändert worden ist, verordnet
    das Bundesministerium für Wirtschaft und Energie im Einvernehmen mit dem
    Bundesministerium für Bildung und Forschung:

    § 1 Staatliche Anerkennung des Ausbildungsberufes

    Der Ausbildungsberuf des Fachinformatikers und der Fachinformatikerin wird nach
    § 4 Absatz 1 des Berufsbildungsgesetzes staatlich anerkannt.

    Berlin, den 28. Februar 2020

    Der Bundesminister für Wirtschaft und Energie
    Peter Altmaier
    """

    # Initialize extractor
    extractor = GermanLegalMetadataExtractor()

    # Extract metadata
    print("🔍 Extracting metadata from sample document...")
    metadata = extractor.extract_metadata(sample_text, "fachinformatiker_vo_2020.txt")

    # Display results
    print("\n📊 EXTRACTED METADATA:")
    print("="*50)
    for key, value in metadata.items():
        if value:
            print(f"{key.upper()}: {value}")

    print(f"\n🎯 CONFIDENCE SCORE: {metadata['confidence_score']}")

    return metadata


def main():
    """Main function for command line usage"""
    import argparse

    parser = argparse.ArgumentParser(description='German Legal Document Metadata Extractor')
    parser.add_argument('input', help='Input text file or directory')
    parser.add_argument('-o', '--output', help='Output file path')
    parser.add_argument('-f', '--format', choices=['json', 'csv'], default='json',
                       help='Output format')
    parser.add_argument('--test', action='store_true', help='Run test with sample data')

    args = parser.parse_args()

    if args.test:
        test_extractor()
        return

    # Initialize extractor
    extractor = GermanLegalMetadataExtractor()

    # Process input
    if Path(args.input).is_file():
        # Single file
        with open(args.input, 'r', encoding='utf-8') as f:
            text = f.read()

        metadata = extractor.extract_metadata(text, args.input)

        if args.output:
            extractor.export_results([metadata], args.output, args.format)
        else:
            print(json.dumps(metadata, indent=2, ensure_ascii=False))

    elif Path(args.input).is_dir():
        # Directory of files
        texts = []
        filenames = []

        for file_path in Path(args.input).glob('*.txt'):
            with open(file_path, 'r', encoding='utf-8') as f:
                texts.append(f.read())
                filenames.append(file_path.name)

        results = extractor.process_batch(texts, filenames)

        if args.output:
            extractor.export_results(results, args.output, args.format)
        else:
            print(json.dumps(results, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    # Run test if called directly
    print("🧪 Testing German Legal Metadata Extractor...")
    test_extractor()